In [1]:
import torch
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from transformers import AutoProcessor, BitsAndBytesConfig, AutoModelForImageTextToText

[2025-06-14 16:00:45,965] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


[2025-06-14 16:00:47,115] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False


In [ ]:
USE_LORA = False 
USE_QLORA = True
SMOL = True

model_id = "HuggingFaceTB/SmolVLM2-256M-Video-Instruct" if SMOL else "HuggingFaceTB/SmolVLM2-2.2B-Instruct"

processor = AutoProcessor.from_pretrained(
    model_id
)

if USE_QLORA or USE_LORA:
    lora_config = LoraConfig(
        r=8,
        lora_alpha=8,
        lora_dropout=0.1,
        target_modules=['down_proj','o_proj','k_proj','q_proj','gate_proj','up_proj','v_proj'],
        use_dora=False if USE_QLORA else True,
        init_lora_weights="gaussian"
    )
    lora_config.inference_mode = False
    if USE_QLORA:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16
        )

    model = AutoModelForImageTextToText.from_pretrained(
        model_id,
        quantization_config=bnb_config if USE_QLORA else None,
        _attn_implementation="flash_attention_2",
        device_map="auto"
    )
    model.add_adapter(lora_config)
    model.enable_adapters()
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, lora_config)
    print(model.get_nb_trainable_parameters())
else:
    model = AutoModelForImageTextToText.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        _attn_implementation="flash_attention_2",
    ).to("cuda")

    # if you'd like to only fine-tune LLM
    # for param in model.model.vision_model.parameters():
    #     param.requires_grad = False

peak_mem = torch.cuda.max_memory_allocated()
print(f"The model as is is holding: {peak_mem / 1024**3:.2f}GB of GPU RAM")

(2884608, 259369536)
The model as is is holding: 0.38GB of GPU RAM


/home/znyd/hacking/edu-cut/.venv/lib/python3.12/site-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/home/znyd/hacking/edu-cut/.venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:167: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
for name, module in model.named_modules():
    print(name)
    # if 'vision' in name.lower() or 'siglip' in name.lower():
    #     print(name)

In [7]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA GeForce RTX 3060. Max memory = 11.638 GB.
1.494 GB of memory reserved.


In [1]:
from datasets import load_dataset

In [2]:
train_ds = load_dataset('json', data_files="../task_fine_tuning/train.jsonl", split='train')
train_ds[0]

Generating train split: 0 examples [00:00, ? examples/s]

{'id': 0,
 'video': 'videos/000_Pi1-b50VHB8.mp4',
 'conversations': [{'from': 'human',
   'value': 'This video segment is a part of a long educational video.\n\nCurrent Segment Transcript: "Okay, hello, can you hear my voice? Yes ma\'am. Okay, so let\'s start."\n\nUser Query: Analyze this educational video and detect irrelevant segments that disrupt the learning process. Identify:\n1. Off-topic discussions (personal stories, unrelated chat, admin talks, etc.)\n2. Silent or non-instructional activities (e.g. instructor drawing silently, pauses, technical setups)\nFor each irrelevant segment, provide:\n- start and end timestamps\n-  brief description with reasoning why it is irrelevant'},
  {'from': 'gpt',
   'value': 'Irrelevant Segment Type: off_topic_discussion \n- Start: 00:00:00\n- End: 00:00:08\n-Reasoning description: The instructor begins the session by checking if their voice is audible and then formally starts the lecture. This initial interaction is a logistical setup for the 

In [ ]:
from torch.nn.utils.rnn import pad_sequence

image_token_id = processor.tokenizer.additional_special_tokens_ids[
    processor.tokenizer.additional_special_tokens.index("<image>")
]

def collate_fn(examples):

    instances = []
    for example in examples:
        prompt = example['conversations'][1]['value']
        query = example['conversations'][0]['value'][8:]

        user_content = [{"type": "text", "text": f"{query}"}]
        user_content.append({"type": "video", "path": example['video']})

        messages = [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": [{"type": "text", "text": f"{prompt}"}]}
        ]
        print(messages)

        instance = processor.apply_chat_template(messages, add_generation_prompt=False,
                                                 tokenize=True, return_dict=True, return_tensors="pt").to("cuda").to(model.dtype)
        instances.append(instance)


    input_ids = pad_sequence(
        [inst["input_ids"].squeeze(0) for inst in instances],
        batch_first=True,
        padding_value=processor.tokenizer.pad_token_id
    )
    attention_mask = pad_sequence(
        [inst["attention_mask"].squeeze(0) for inst in instances],
        batch_first=True,
        padding_value=0
    )
    labels = pad_sequence(
        [inst["input_ids"].squeeze(0).clone() for inst in instances],
        batch_first=True,
        padding_value=-100
    )

    labels[labels == image_token_id] = -100

    out = {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }


    # Step 1: figure out maximum frames, height, width across the batch
    pvs = [inst["pixel_values"].squeeze(0) for inst in instances if "pixel_values" in inst]
    if pvs:  # there is at least one non-None pixel_values
        max_frames = max(pv.shape[0] for pv in pvs)
        max_h = max(pv.shape[-2] for pv in pvs)
        max_w = max(pv.shape[-1] for pv in pvs)
    else:
        max_h = max_w = processor.video_size['longest_edge']
        max_frames = 1

    padded_pixel_values_list = []
    for ex in instances:
        pv = ex.get("pixel_values", None).squeeze(0)

        if pv is None:
            # text-only => fill pixel data + mask with zeros
            shape_pv = (max_frames, 3, max_h, max_w)
            padded_pv = torch.zeros(shape_pv, dtype=torch.float32)
        else:
            f, c, h, w = pv.shape
            # Prepare final storage
            padded_pv = torch.zeros(
                (max_frames, c, max_h, max_w),
                dtype=pv.dtype,
                device=pv.device
            )
            padded_pv[:f, :, :h, :w] = pv
        padded_pixel_values_list.append(padded_pv)

    out["pixel_values"] = torch.stack(padded_pixel_values_list, dim=0)
    return out


[{'role': 'user', 'content': [{'type': 'text', 'text': 'eo segment is a part of a long educational video.\n\nCurrent Segment Transcript: "Okay, hello, can you hear my voice? Yes ma\'am. Okay, so let\'s start."\n\nUser Query: Analyze this educational video and detect irrelevant segments that disrupt the learning process. Identify:\n1. Off-topic discussions (personal stories, unrelated chat, admin talks, etc.)\n2. Silent or non-instructional activities (e.g. instructor drawing silently, pauses, technical setups)\nFor each irrelevant segment, provide:\n- start and end timestamps\n-  brief description with reasoning why it is irrelevant'}, {'type': 'video', 'path': 'videos/000_Pi1-b50VHB8.mp4'}]}, {'role': 'assistant', 'content': [{'type': 'text', 'text': 'Irrelevant Segment Type: off_topic_discussion \n- Start: 00:00:00\n- End: 00:00:08\n-Reasoning description: The instructor begins the session by checking if their voice is audible and then formally starts the lecture. This initial intera

NameError: name 'processor' is not defined

# Training


In [7]:
from transformers import TrainingArguments, Trainer

model_name = model_id.split("/")[-1]

training_args = TrainingArguments(
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    warmup_steps=50,
    learning_rate=1e-4,
    weight_decay=0.01,
    logging_steps=25,
    save_strategy="steps",
    save_steps=250,
    save_total_limit=1,
    optim="adamw_torch", # for 8-bit, keep paged_adamw_8bit, else adamw_hf
    bf16=True,
    output_dir=f"./{model_name}-dense-caption",
    remove_unused_columns=False,
    report_to="tensorboard",
    dataloader_pin_memory=False
)

In [8]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=collate_fn,
    train_dataset=train_ds,
)

No label_names provided for model class `PeftModel`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:
trainer.train()

[{'role': 'user', 'content': [{'type': 'text', 'text': 'This video segment is a part of a long educational video on Control Systems: Block Diagram Reduction Rules and Techniques\nPrevious clips Context: The instructor has just introduced the concept of block diagram reduction, showing a complex diagram and explaining the goal is to simplify it.\n\nCurrent Segment Transcript: "can be given to you, and the task will be to reduce this block, reduce the whole block diagram to a single block, okay? So as you can see that this block diagram has several blocks in it, right? But your task is to reduce the whole, the total, all of the blocks to a single block"\n\nUser Query: Based on the provided context, I need you to do two things for this video segment: \n1. Provide a detailed analysis of the visual information and the educational concept being taught.\n2. Pinpoint the exact start and end times for this specific segment.'}, {'type': 'video', 'path': 'videos/002_tJZZXoTHRFg.mp4'}]}, {'role': 

In [ ]:
# Load the TensorBoard extension
%load_ext tensorboard

# Start TensorBoard, pointing it to your output directory
%tensorboard --logdir ./{model_name}-dense-caption